<div class='bar_title'></div>

*Simulation for Decision Making (S4DM)*

# Generating Input Data

Summer Semester 26

Prof. Dr. Gunther Gust <br>
Chair for Enterprise AI <br>
Data Driven Decisions Group <br>
Center for Artificial Intelligence and Data Science (CAIDAS)

<img src="images/d3.png" style="width:20%; float:left;" />

<img src="images/CAIDASlogo.png" style="width:20%; float:left;" />

# Where we are

<img src="images/simulation_study_steps_input.png" style="width:80%; float:center;" />

**Last lecture**: how to *choose* a distribution that fits the data.

**This lecture**: how to *generate* random samples from that distribution during a simulation run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import simpy
import random
from scipy import stats

rng_global = np.random.default_rng()  # NumPy's modern RNG

# Hook — same model, different seed

We rerun the **same** SimPy mensa simulation twice. The only difference: the random seed.

In [ ]:
# --- Resource class ---
class Mensa:
    def __init__(self, env, num_counters, mean_service):
        self.env = env
        self.counter = simpy.Resource(env, capacity=num_counters)
        self.mean_service = mean_service

    def serve(self):
        yield self.env.timeout(random.expovariate(1 / self.mean_service))


# --- Entity class ---
class Student:
    def __init__(self, env, name, mensa, waits):
        self.env = env
        self.name = name
        self.mensa = mensa
        self.waits = waits

    def run(self):
        arrive = self.env.now
        with self.mensa.counter.request() as req:
            yield req
            self.waits.append(self.env.now - arrive)
            yield self.env.process(self.mensa.serve())


# --- Generator function ---
def student_generator(env, mensa, waits, mean_interarrival):
    i = 0
    while True:
        yield env.timeout(random.expovariate(1 / mean_interarrival))
        c = Student(env, f'C{i}', mensa, waits)
        env.process(c.run())
        i += 1


# --- Run block ---
def run_mensa(seed, mean_interarrival=2.0, mean_service=1.5,
              num_counters=1, sim_time=200):
    random.seed(seed)
    env = simpy.Environment()
    mensa = Mensa(env, num_counters=num_counters, mean_service=mean_service)
    waits = []
    env.process(student_generator(env, mensa, waits, mean_interarrival))
    env.run(until=sim_time)
    return np.mean(waits)


# Same model, different seeds → different average waiting times
for seed in [1, 7, 42, 123, 2024]:
    print(f'seed={seed:5d}  →  avg wait = {run_mensa(seed):.2f} min')

# What we just observed

Same model, same parameters, same code — yet the **average waiting time differs by minutes** depending on a single integer: the seed.

Every result your simulation produces is downstream of a stream of *random numbers*. If we change where that stream starts, every customer arrives at a slightly different time, every service takes a slightly different duration, and the queue evolves down a different path.

**Two practical consequences:**

- **Reproducibility** — without a fixed seed, you can't compare runs, debug behavior, or share results
- **Comparison fairness** — if you compare two designs with *different* random streams, you can't tell whether the difference is due to the *design* or just the *randomness*

Before we can trust simulation results, we need to understand where these random numbers come from and how to control them.

# Generating input data is a *two-step* process

Every time SimPy needs an inter-arrival time, a service time, or a demand quantity, the computer goes through **two stages** under the hood:

<br>

$$\underbrace{U \sim \text{Uniform}(0, 1)}_{\text{Step 1 output}} \;\;\longrightarrow\;\; \underbrace{X \sim F}_{\text{Step 2 output — the sample we want}}$$

<br>

with
- $U$ — a **pseudo-random number** uniformly distributed on $[0, 1]$
- $X$ — a **random variate** drawn from the target distribution with CDF $F$ (e.g. Exponential, Normal, Triangular, ...)

<br>

| Step | What happens | Question for today |
|---|---|---|
| **1. Pseudo-random number** | A deterministic algorithm produces $U$ — a number that *looks* uniform on $[0, 1]$ | *Where does $U$ come from? What does `random.seed(42)` actually do?* |
| **2. Random variate** | Transform $U$ into a sample $X$ from the desired distribution | *How does NumPy turn a $U$ into an Exponential sample?* |

Understanding both steps is what separates a *user* of random numbers from a *modeller* who can trust, debug, and extend their simulations.

---
# Part 1 — Generating uniform pseudo-random numbers

# Why *pseudo*-random?

Computers are **deterministic** — they cannot produce truly random numbers without external help.

**Historical attempts at *real* randomness:**
- Throwing dice or dealing cards
- Reading digits of π
- Spinning discs, ERNIE (UK 1957: an electronic random number machine for premium bonds)
- Counting gamma rays

All impractical for simulation: **slow**, **non-reproducible**, hard to feed into a program.

# What we actually want

Algorithmically generated numbers $U_1, U_2, \ldots \in [0, 1]$ that *behave* random:

| Property | Why we need it |
|---|---|
| **Uniformity** | $U_i \sim \text{Uniform}(0, 1)$ — no value is favored |
| **Independence** | Successive values are not correlated |
| **Long period** | The sequence does not repeat for a *very* long time |
| **Reproducibility** | Same seed → same sequence (debugging, comparison) |
| **Speed** | Millions of samples per simulation |

**Trade-off:** we give up *true* randomness to gain reproducibility — and reproducibility is what makes scientific simulation possible.

# The Linear Congruential Method (LCM)

The classical workhorse — simple enough to compute by hand:

$$X_{n+1} = (a \cdot X_n + c) \bmod m$$

$$U_n = \frac{X_n}{m}$$

with:
- $X_0$ — the **seed** (starting value)
- $a$ — multiplier
- $c$ — increment
- $m$ — modulus

Every $X_n \in \{0, 1, \ldots, m-1\}$, so dividing by $m$ gives a value in $[0, 1)$.

# LCM — example by hand

Let $a = 5$, $c = 3$, $m = 16$, $X_0 = 7$.

$X_1 = (5 \cdot 7 + 3) \bmod 16 = 38 \bmod 16 = 6$

$X_2 = (5 \cdot 6 + 3) \bmod 16 = 33 \bmod 16 = 1$

$X_3 = (5 \cdot 1 + 3) \bmod 16 = 8$

$X_4 = (5 \cdot 8 + 3) \bmod 16 = 11$

$\ldots$

After at most $m = 16$ steps, the sequence **must** repeat — this is the **period**.

In [ ]:
# LCM in code — let's compute the period
def lcm(seed, a, c, m, n):
    x = seed
    out = []
    for _ in range(n):
        x = (a * x + c) % m
        out.append(x)
    return out

seq = lcm(seed=7, a=5, c=3, m=16, n=20)
print('Sequence:', seq)
print(f'Period:    {seq[:4]} == {seq[16:20]} ? {seq[:4] == seq[16:20]}')
# After m = 16 steps, the sequence repeats from the beginning — full period of 16.

# A *bad* choice of parameters

Not every $(a, c, m)$ produces a long period. With poor choices the sequence may repeat after only a handful of values.

In [ ]:
# Try a poor choice: a=2, c=0, m=16
bad = lcm(seed=4, a=2, c=0, m=16, n=10)
print('Bad choice sequence:', bad)
# The sequence collapses to 0 — useless as a random source

# What makes a *good* LCG?

There are formal **full-period conditions** (Hull & Dobell 1962). They guarantee the LCG cycles through *all* $m$ values before repeating.

*We won't derive them today* — see the appendix for details. The takeaway is: choosing $a, c, m$ is not free, and bad choices have caused real bugs in production simulators.

**Modern reality**: nobody hand-tunes LCG parameters anymore. We use battle-tested generators like the **Mersenne Twister** and **PCG64**.

# Modern default: NumPy's `default_rng`

NumPy's recommended generator is **PCG64** (Permuted Congruential Generator):

- Period $\approx 2^{128} \approx 3.4 \times 10^{38}$
- Statistically excellent — passes modern test batteries (TestU01, PractRand)
- Fast, reproducible, parallel-safe

**For perspective on "long enough":**

| Generator | Period |
|---|---|
| **PCG64** | $\sim 10^{38}$ — at 1 billion samples/sec it would take $\sim 10^{22}$ years to exhaust |
| **Mersenne Twister** | $\sim 10^{6001}$ — *vastly* more than the $\sim 10^{80}$ atoms in the observable universe |

Either way: you will never run out.

**The only thing you need to remember:**

In [ ]:
# Same seed → identical sequence (reproducibility)
rng_a = np.random.default_rng(seed=42)
rng_b = np.random.default_rng(seed=42)

print('rng_a:', rng_a.random(5))
print('rng_b:', rng_b.random(5))
print('Identical?', np.array_equal(rng_a.random(100), rng_b.random(100)))

In [ ]:
# Sampling from distributions — one rng object, many distribution methods
rng = np.random.default_rng(seed=42)

print('Uniform[0,1) :', rng.random(3))
print('Exponential  :', rng.exponential(scale=2.0, size=3))
print('Normal       :', rng.normal(loc=0, scale=1, size=3))
print('Triangular   :', rng.triangular(left=0, mode=5, right=10, size=3))

# Random-number streams & CRN

Real simulations consume random numbers from **multiple sources**: arrivals, service times, machine breakdowns, customer choices, ...

If they all share **one global stream**, a tiny change in any one of them shifts the entire downstream sequence — making fair comparison between system designs impossible.

**Solution: dedicated streams.** Assign one independent RNG per source of randomness.

# Common Random Numbers (CRN)

When comparing two system designs (e.g. *1 cashier* vs *2 cashiers*), we want both to face the **same** sequence of arrivals and service-time draws — so any difference in waiting time is attributable to the **design**, not to random noise.

**CRN** = use the same random-number streams for the same purpose in each variant.

*Result: dramatic variance reduction when comparing alternatives — a topic we'll revisit in the **output analysis** lecture.*

In [ ]:
# --- Resource class ---
class Mensa:
    def __init__(self, env, num_cashiers, mean_service, svc_rng):
        self.env = env
        self.counter = simpy.Resource(env, capacity=num_cashiers)
        self.mean_service = mean_service
        self.svc_rng = svc_rng        # dedicated service-time stream

    def serve(self):
        yield self.env.timeout(self.svc_rng.exponential(self.mean_service))


# --- Entity class ---
class Student:
    def __init__(self, env, name, mensa, waits):
        self.env = env
        self.name = name
        self.mensa = mensa
        self.waits = waits

    def run(self):
        arrive = self.env.now
        with self.mensa.counter.request() as req:
            yield req
            self.waits.append(self.env.now - arrive)
            yield self.env.process(self.mensa.serve())


# --- Generator function ---
def student_generator(env, mensa, waits, mean_interarrival, arr_rng):
    i = 0
    while True:
        yield env.timeout(arr_rng.exponential(mean_interarrival))
        env.process(Student(env, f'S{i}', mensa, waits).run())
        i += 1


# --- Run block ---
def compare_cashiers(num_cashiers, arr_seed, svc_seed,
                     mean_interarrival=1.0, mean_service=1.5, sim_time=20):
    arr_rng = np.random.default_rng(arr_seed)
    svc_rng = np.random.default_rng(svc_seed)
    env = simpy.Environment()
    mensa = Mensa(env, num_cashiers=num_cashiers,
                  mean_service=mean_service, svc_rng=svc_rng)
    waits = []
    env.process(student_generator(env, mensa, waits, mean_interarrival, arr_rng))
    env.run(until=sim_time)
    return np.mean(waits)

In [ ]:
# Bad choice: different seeds for the two designs → unfair comparison
w1 = compare_cashiers(num_cashiers=1, arr_seed=0, svc_seed=0)
w2 = compare_cashiers(num_cashiers=2, arr_seed=1, svc_seed=1)
print(f'1 cashier  (seed 0): avg wait = {w1:.2f} min')
print(f'2 cashiers (seed 1): avg wait = {w2:.2f} min')
print(f'"Improvement":             {w1 - w2:+.2f} min  ← 2 cashiers look WORSE!')

In [ ]:
# CRN: same seeds for both designs → fair comparison
w1 = compare_cashiers(num_cashiers=1, arr_seed=0, svc_seed=0)
w2 = compare_cashiers(num_cashiers=2, arr_seed=0, svc_seed=0)
print(f'1 cashier  (seed 0): avg wait = {w1:.2f} min')
print(f'2 cashiers (seed 0): avg wait = {w2:.2f} min')
print(f'Improvement:               {w1 - w2:+.2f} min  ← 2 cashiers correctly look better')

# 🧠 Mentimeter — your turn

➡️ Open the Mentimeter and answer the question on screen.

---
# Part 2 — From Uniform to anything

# We have $U \sim \text{Uniform}(0,1)$. Now what?

All distributions in NumPy ultimately rest on this foundation. The question is:

> Given a stream of uniform numbers, how do we generate samples from an Exponential, Triangular, Normal, or even an empirical distribution?

**Three families of techniques:**

| Method | When to use |
|---|---|
| **Inverse transform** | CDF is invertible (Exponential, Weibull, Triangular, empirical) |
| **Acceptance-rejection** | CDF is hard to invert (Normal, Gamma, Beta) |
| **Composition / convolution** | Sum or mixture of simpler distributions (Erlang) |

We focus on **inverse transform** — it's the most general, most used, and pedagogically the most important.

# Inverse transform — geometric intuition

Picture the CDF $F(x)$ of the distribution we want to sample from.

1. Drop a uniform random value $u$ onto the **y-axis** ($u \in [0, 1]$)
2. Draw a horizontal line until it hits the curve $F$
3. Drop **down** to the x-axis — the landing point is your sample $x$

**Why does this produce the right density?** Steeper parts of the CDF (where the PDF is high) are hit by more horizontal lines → more samples land there. Flat parts get fewer samples. The mechanism *automatically* matches the desired density.

In [ ]:
# Visualize the inverse-transform on an Exponential CDF
rng = np.random.default_rng(0)
lam = 1.0
x = np.linspace(0, 5, 400)
F = 1 - np.exp(-lam * x)

u_samples = rng.random(5)
x_samples = -np.log(1 - u_samples) / lam

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, F, color='#1f77b4', linewidth=2.5, label='CDF $F(x)$')
for u, xs in zip(u_samples, x_samples):
    ax.hlines(u, 0, xs, color='#ff7f0e', linewidth=1, linestyle='--')
    ax.vlines(xs, 0, u, color='#ff7f0e', linewidth=1, linestyle='--')
    ax.plot(xs, u, 'o', color='#ff7f0e')
ax.set_xlabel('x'); ax.set_ylabel('F(x)')
ax.set_title('Inverse transform: u → x')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(); plt.tight_layout(); plt.show()

# Inverse transform — algorithm

**Theorem.** If $F$ is a continuous CDF and $U \sim \text{Uniform}(0, 1)$, then

$$X = F^{-1}(U)$$

is a random variable with CDF $F$.

**Algorithm in 3 lines:**

1. Draw $U \sim \text{Uniform}(0, 1)$
2. Solve $F(X) = U$ for $X$ — i.e. compute $X = F^{-1}(U)$
3. Return $X$

# Trivial example — Uniform $(a, b)$

Before tackling Exponential, let's apply the recipe to the simplest CDF possible.

**CDF:** $F(x) = \dfrac{x - a}{b - a}$ for $x \in [a, b]$

**Set $F(X) = U$ and solve:**

$$\frac{X - a}{b - a} = U \quad\Longrightarrow\quad \boxed{X = a + (b - a) \cdot U}$$

One uniform draw, one linear scaling — done. The full recipe in its simplest form.

# Worked example — Exponential

Exponential with rate $\lambda$:

$$F(x) = 1 - e^{-\lambda x}, \quad x \geq 0$$

**Set $F(X) = U$ and solve:**

$$1 - e^{-\lambda X} = U$$
$$e^{-\lambda X} = 1 - U$$
$$-\lambda X = \ln(1 - U)$$
$$\boxed{X = -\frac{1}{\lambda} \ln(1 - U)}$$

Since $1 - U$ is itself uniform on $[0, 1)$, this is often written as $X = -\ln(U) / \lambda$.

In [ ]:
# Implement the Exponential sampler from scratch and validate
rng = np.random.default_rng(42)
lam = 0.5
n = 10_000

# Our hand-rolled inverse-transform sampler
u = rng.random(n)
x_ours = -np.log(1 - u) / lam

# NumPy's built-in for comparison
x_numpy = np.random.default_rng(42).exponential(scale=1/lam, size=n)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(x_ours, bins=60, alpha=0.6, label='Ours: $-\ln(1-U)/\lambda$', color='#1f77b4', density=True)
ax.hist(x_numpy, bins=60, alpha=0.4, label='NumPy .exponential()', color='#ff7f0e', density=True)
ax.set_xlabel('x'); ax.set_ylabel('Density')
ax.set_title(f'Exponential samples (λ = {lam}), n = {n}')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(); plt.tight_layout(); plt.show()

# Worked example — Triangular

Triangular with min $a$, mode $c$, max $b$ (here $a = 0$, $b = 1$, $c$ free):

$$F(x) = \begin{cases} \frac{x^2}{c} & 0 \leq x \leq c \\[6pt] 1 - \frac{(1-x)^2}{1-c} & c < x \leq 1 \end{cases}$$

Inverting piecewise:

$$X = \begin{cases} \sqrt{c \cdot U} & U \leq c \\ 1 - \sqrt{(1-c)(1-U)} & U > c \end{cases}$$

In [ ]:
# Triangular inverse-transform sampler
def sample_triangular(rng, n, c=0.5):
    u = rng.random(n)
    return np.where(u <= c, np.sqrt(c * u), 1 - np.sqrt((1 - c) * (1 - u)))

rng = np.random.default_rng(0)
c = 0.7
samples = sample_triangular(rng, 20_000, c=c)

# Reference: theoretical PDF of Triangular(0, 1, mode=c)
x = np.linspace(0, 1, 400)
pdf = np.where(x <= c, 2 * x / c, 2 * (1 - x) / (1 - c))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(samples, bins=60, color='#1f77b4', edgecolor='black', linewidth=0.4,
        density=True, label='Inverse-transform samples')
ax.plot(x, pdf, color='#ff7f0e', linewidth=2.5, label='Theoretical PDF')
ax.set_xlabel('x'); ax.set_ylabel('Density')
ax.set_title(f'Triangular(0, 1, mode={c}) — sampled via inverse transform')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(); plt.tight_layout(); plt.show()

# Sampling from an *empirical* CDF

What if **no parametric distribution fits** your data? Use the data itself.

**Procedure:**

1. Sort the observations $x_{(1)} \leq x_{(2)} \leq \ldots \leq x_{(n)}$
2. Build the empirical CDF: $F_n(x_{(i)}) = i/n$
3. Smooth via **piecewise-linear interpolation** between the points $(x_{(i)}, i/n)$
4. Sample by inverting this piecewise-linear CDF

This is exactly the inverse-transform method applied to a CDF you built from data.

In [ ]:
# Empirical CDF sampling — end-to-end demo
# Pretend these are 8 observed service times we collected
observed = np.array([1.5, 2.2, 3.0, 3.8, 4.7, 5.5, 7.0, 9.0])

sorted_x = np.sort(observed)
n = len(sorted_x)

# Piecewise-linear empirical CDF for sampling.
# Anchor: F(x_min) = 0, F(x_max) = 1, with F(x_(i)) = (i-1)/(n-1).
F_pl = np.arange(n) / (n - 1)

# Sample by inverse-transform on the piecewise-linear CDF
rng = np.random.default_rng(1)
u = rng.random(5_000)
samples = np.interp(u, F_pl, sorted_x)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left panel: piecewise-linear empirical CDF
axes[0].plot(sorted_x, F_pl, color='#ff7f0e', linewidth=2,
             label='Piecewise-linear CDF')
axes[0].plot(sorted_x, F_pl, 'o', color='#ff7f0e', markersize=7, label='Observations')
axes[0].set_xlabel('Service time'); axes[0].set_ylabel('F(x)')
axes[0].set_title('Empirical CDF from data')
axes[0].legend(loc='lower right')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Right panel: samples + original observations as ticks
axes[1].hist(samples, bins=40, color='#1f77b4', edgecolor='black', linewidth=0.4, density=True)
axes[1].plot(sorted_x, [0.005]*len(sorted_x), '|', color='#ff7f0e', markersize=14,
             markeredgewidth=2, label='Original data')
axes[1].set_xlabel('Service time'); axes[1].set_ylabel('Density')
axes[1].set_title('5,000 samples drawn from the empirical CDF')
axes[1].legend()
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

# When inverse transform fails

Some distributions have **no closed-form CDF** — and therefore no closed-form $F^{-1}$:

- **Normal**: CDF is the error function $\Phi(x)$
- **Gamma** (general shape parameter)
- **Beta**

**Workarounds:**
- **Convolution** — when the target is a *sum* of simpler variates
- **Acceptance-Rejection** — sample from an easy *envelope* distribution and accept/reject
- **Specialized methods** — Box-Muller for Normal, Marsaglia-Tsang for Gamma

**Good news:** NumPy's `default_rng()` hides all of this from you. But now you know it's happening underneath.

# Convolution — sums of simpler variates

If the target distribution is the **sum of independent simpler ones**, just add them up — no inversion, no rejection.

**Classic example: Erlang $(k, \lambda)$** = sum of $k$ independent Exponential $(\lambda)$ variables

```python
sample = sum(-np.log(rng.random()) / lam for _ in range(k))
```

**Other examples:**

| Target | = sum of |
|---|---|
| Binomial $(n, p)$ | $n$ Bernoulli $(p)$ trials |
| Negative Binomial $(k, p)$ | $k$ Geometric $(p)$ |
| Erlang $(k, \lambda)$ | $k$ Exponential $(\lambda)$ |
| Chi-square $(k)$ | $k$ squared standard Normals |

The name comes from the fact that the PDF of a sum of independent variables is the **convolution** of their PDFs.

# Acceptance-Rejection — the intuition

To sample from a complicated PDF $f(x)$ on a bounded interval $[a, b]$:

1. Find a height $h$ that bounds $f$ from above: $f(x) \leq h$ for all $x \in [a, b]$. The simplest choice is $h = \max_x f(x)$.
2. Sample a candidate $X \sim \text{Uniform}(a, b)$ and an independent height $Y \sim \text{Uniform}(0, h)$.
3. **Accept the candidate $X$** if $Y \leq f(X)$ — i.e. the dart $(X, Y)$ landed *under* the curve $f$. Otherwise **reject** and try again.

**Why does this work?** Step 2 throws a *uniform dart* into the rectangle $[a, b] \times [0, h]$. Step 3 keeps only darts that land under $f$. By construction, the accepted darts are uniformly distributed under $f$, so their $x$-coordinates follow the desired density.

**The price you pay:** rejected samples. The fraction kept equals $\frac{\text{area under } f}{\text{area of rectangle}} = \frac{1}{h(b - a)}$ — the closer the rectangle hugs the curve, the higher the acceptance rate.

*The general version uses an arbitrary envelope $g(x)$ instead of a flat $h$ — see the appendix and Banks et al. (2013) Ch. 8 for details.*

In [ ]:
# Acceptance-Rejection — visual illustration
# Target: half-Normal (Normal restricted to [0, 4]) — has no closed-form inverse CDF
# Envelope: uniform rectangle of height M over [0, 4]

from scipy.stats import norm

rng = np.random.default_rng(7)
x_lo, x_hi = 0, 4
f = lambda x: 2 * norm.pdf(x)              # target PDF (half-Normal)
M = f(0)                                   # envelope height (max of f)

# Generate candidate points uniformly in the bounding box
n_candidates = 600
x_cand = rng.uniform(x_lo, x_hi, n_candidates)
y_cand = rng.uniform(0, M, n_candidates)
accepted = y_cand <= f(x_cand)

# Plot
x_grid = np.linspace(x_lo, x_hi, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Left panel: the geometry
axes[0].plot(x_grid, f(x_grid), color='#1f77b4', linewidth=2.5, label='Target $f(x)$')
axes[0].axhline(M, color='#ff7f0e', linewidth=2, linestyle='--', label=f'Envelope $M \\cdot g(x) = {M:.2f}$')
axes[0].fill_between(x_grid, 0, f(x_grid), color='#1f77b4', alpha=0.15)
axes[0].scatter(x_cand[accepted], y_cand[accepted], color='#2ca02c', s=10,
                label=f'Accepted ({accepted.sum()})', alpha=0.75)
axes[0].scatter(x_cand[~accepted], y_cand[~accepted], color='#d62728', s=10,
                label=f'Rejected ({(~accepted).sum()})', alpha=0.45, marker='x')
axes[0].set_xlabel('x'); axes[0].set_ylabel('density')
axes[0].set_title('Step 1 — sample uniformly under the envelope')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Right panel: the accepted x-values form the target distribution
axes[1].hist(x_cand[accepted], bins=25, color='#2ca02c', edgecolor='black',
             linewidth=0.4, density=True, alpha=0.7, label='Accepted samples')
axes[1].plot(x_grid, f(x_grid), color='#1f77b4', linewidth=2.5, label='Target $f(x)$')
axes[1].set_xlabel('x'); axes[1].set_ylabel('density')
axes[1].set_title(f'Step 2 — keep only $x$ of accepted points  (rate = {accepted.mean():.0%})')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

# Recap

<br>

| Topic | Key idea |
|---|---|
| Pseudo-random | Deterministic algorithm — *seed* controls the sequence |
| LCM | Classical example, illustrates *period* and the cost of bad parameters |
| `default_rng()` | Modern default — fast, long period, reproducible, parallel-safe |
| Streams & CRN | Separate streams per random source enable fair comparisons |
| Inverse transform | Universal recipe: $X = F^{-1}(U)$ |
| Empirical CDF | Sample directly from data when no fit applies |
| Convolution | Sum of independent simpler variates (e.g. Erlang from Exponentials) |
| Acceptance-Rejection | Fallback for hard-to-invert CDFs (Normal, Gamma, ...) |

# Next session

<img src="images/simulation_study_steps_vandv.png" style="width:80%; float:center;" />

**Verification and Validation** — your simulation now produces reproducible random inputs. But:

- Does the model do what you *intended* it to do? *(verification)*
- Does the model faithfully represent the *real* system? *(validation)*

We will look at techniques to answer both questions before we trust simulation output for decision making.

# 🧠 Mentimeter — your turn

➡️ Open the Mentimeter and answer the question on screen.

---
# Appendix

# Appendix — Hull-Dobell full-period conditions

An LCG $X_{n+1} = (a X_n + c) \bmod m$ has **full period** $m$ if and only if:

1. $\gcd(c, m) = 1$ — the only positive integer that exactly divides both $m$ and $c$ is 1
2. For every prime $q$ that divides $m$: $q$ divides $a - 1$
3. If $4 \mid m$, then $4 \mid (a - 1)$

Reference: Hull & Dobell (1962). See Banks et al. (2013) Ch. 7 for proof.

# Appendix — Box-Muller (Normal sampling)

Given two independent uniforms $U_1, U_2 \in (0, 1)$, the Box-Muller transform produces two independent standard Normal random variables:

$$Z_1 = \sqrt{-2 \ln U_1} \, \cos(2\pi U_2), \qquad Z_2 = \sqrt{-2 \ln U_1} \, \sin(2\pi U_2)$$

It is a classical example of a **direct transformation** that bypasses inverse transform entirely — turning one uniform pair into one Normal pair with no rejection.

---

## Proof

Define polar coordinates

$$R = \sqrt{-2 \ln U_1}, \qquad \Theta = 2\pi U_2$$

so that $Z_1 = R \cos \Theta$ and $Z_2 = R \sin \Theta$.

**Step 1 — Distribution of $\Theta$.** Trivially, $\Theta \sim \text{Uniform}(0, 2\pi)$ with density $f_\Theta(\theta) = \frac{1}{2\pi}$.

**Step 2 — Distribution of $R$.** For $r > 0$:

$$P(R \leq r) = P\!\left(\sqrt{-2 \ln U_1} \leq r\right) = P\!\left(U_1 \geq e^{-r^2/2}\right) = 1 - e^{-r^2/2}.$$

This is the **Rayleigh** CDF, with density $f_R(r) = r \, e^{-r^2/2}$ for $r \geq 0$.

**Step 3 — Independence.** $R$ depends only on $U_1$, $\Theta$ only on $U_2$, and $U_1 \perp U_2$. Hence the joint density is

$$f_{R, \Theta}(r, \theta) = r \, e^{-r^2/2} \cdot \frac{1}{2\pi}.$$

**Step 4 — Change of variables to $(Z_1, Z_2)$.** The Jacobian of the polar-to-Cartesian map $(r, \theta) \mapsto (z_1, z_2)$ is

$$\left| \frac{\partial(z_1, z_2)}{\partial(r, \theta)} \right| = \begin{vmatrix} \cos\theta & -r\sin\theta \\ \sin\theta & \,r\cos\theta \end{vmatrix} = r.$$

Therefore

$$f_{Z_1, Z_2}(z_1, z_2) = \frac{f_{R, \Theta}(r, \theta)}{r} = \frac{1}{2\pi} \, e^{-(z_1^2 + z_2^2)/2}$$

since $r^2 = z_1^2 + z_2^2$.

**Step 5 — Factorization.**

$$f_{Z_1, Z_2}(z_1, z_2) = \frac{1}{\sqrt{2\pi}} e^{-z_1^2/2} \cdot \frac{1}{\sqrt{2\pi}} e^{-z_2^2/2} = \varphi(z_1) \cdot \varphi(z_2).$$

The joint density factors into two standard Normal PDFs, so $Z_1$ and $Z_2$ are independent $\mathcal{N}(0, 1)$ random variables. $\blacksquare$

---

## Geometric intuition

The 2D standard Normal is **rotationally symmetric**: its joint density depends only on $z_1^2 + z_2^2$. Box-Muller exploits this:

- $\Theta$ is uniform on the circle (matching the rotational symmetry)
- $R$ is drawn so that $R^2$ is exponential — exactly the distribution of the squared length of a 2D standard Normal vector

Projecting the resulting random point $(R, \Theta)$ onto the $x$- and $y$-axes yields two independent univariate Normals.

# Appendix — References for modern RNGs

**NumPy documentation**
- [PCG64 bit generator](https://numpy.org/doc/stable/reference/random/bit_generators/pcg64.html)
- [`default_rng()` Generator API](https://numpy.org/doc/stable/reference/random/generator.html)

**PCG**
- O'Neill, M. E. (2014). *PCG: A Family of Simple Fast Space-Efficient Statistically Good Algorithms for Random Number Generation.* [Technical report](https://www.pcg-random.org/paper.html)

**Mersenne Twister**
- Matsumoto, M. & Nishimura, T. (1998). *Mersenne Twister: A 623-dimensionally equidistributed uniform pseudo-random number generator.* ACM Trans. Model. Comput. Simul., 8(1), 3–30. [DOI: 10.1145/272991.272995](https://doi.org/10.1145/272991.272995)